# MDM V2 Validation & Performance Analysis

This notebook provides interactive exploration of the MDM V2 validation pipeline:
1. Load best configuration from parameter sweep
2. Run V2 and Classic engines
3. Validate signal match rates (PERF-03)
4. Compare performance across strategies (PERF-01, PERF-02)
5. Visualize with multi-panel dashboard

In [ ]:
import sys
import os

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from analysis.validate_v2 import (
    load_best_config, validate_match_rates, build_comparison_table,
    generate_dashboard, _compute_classic_equity, _compute_buy_and_hold_metrics,
    ANALYSIS_START, TRAIN_START, TRAIN_END, HELDOUT_START, HELDOUT_END
)
from strategies.mdm_v2.config import MDMV2Config
from strategies.mdm_v2.mdm_v2_engine import MDMV2Engine
from strategies.mdm_v2.performance import V2PerformanceAnalyzer, check_degradation
from strategies.mdm_classic.mdm_engine import MDMEngine
from core.signal_comparator import extract_model_signals, compare_signals
from core.data_loader import DataLoader
from core.signal_loader import load_signal_fixture

print('All imports successful')

## 1. Load Configuration and Data

In [ ]:
# Load best config from sweep results
sweep_csv = os.path.join(project_root, 'output', 'sweep_results.csv')
config = load_best_config(sweep_csv)

print('Best Configuration Parameters:')
print(f'  Name: {config.name}')
print(f'  correction_threshold: {config.correction_threshold}')
print(f'  ftd_min_rally_day: {config.ftd_min_rally_day}')
print(f'  ftd_max_rally_day: {config.ftd_max_rally_day}')
print(f'  ftd_min_price_gain: {config.ftd_min_price_gain}')
print(f'  dd_cash_threshold: {config.dd_cash_threshold}')
print(f'  dd_window_size: {config.dd_window_size}')
print(f'  stop_loss_pct: {config.stop_loss_pct}')
print(f'  ma10_cash_enabled: {config.ma10_cash_enabled}')
print(f'  ma50_sell_enabled: {config.ma50_sell_enabled}')

# Load data
loader = DataLoader('nasdaq', data_dir=project_root)
df = loader.load(start_date='2017-01-01')
print(f'\nNASDAQ data: {len(df)} rows ({df["date"].min().date()} to {df["date"].max().date()})')

# Load published signals
signals_path = os.path.join(project_root, 'data', 'signals', 'nasdaq_signals.csv')
published_signals = load_signal_fixture(signals_path)
print(f'Published signals: {len(published_signals)} signals')
published_signals.head(10)

## 2. Run Engines

In [ ]:
# Run V2 engine
v2_engine = MDMV2Engine(config)
v2_results = v2_engine.run(df)
v2_trades = v2_engine.get_trades()
print(f'V2 Engine: {len(v2_trades)} trades')

# Run Classic engine
classic_engine = MDMEngine()
classic_results = classic_engine.run(df)
classic_trades = classic_engine.get_trades()
print(f'Classic Engine: {len(classic_trades)} trades')

# Show V2 trade summary
v2_summary = v2_engine.summary()
print(f'\nV2 Summary:')
for k, v in v2_summary.items():
    print(f'  {k}: {v}')

## 3. Signal Match Validation (PERF-03)

Compare model-generated signals against published signal history for both training and held-out periods.

In [ ]:
# Filter to analysis period for match rate computation
v2_analysis = v2_results[v2_results['date'] >= pd.Timestamp(ANALYSIS_START)].copy()

validation = validate_match_rates(v2_analysis, published_signals)

print('SIGNAL MATCH VALIDATION')
print('=' * 50)
print(f'\nTraining Period (2019-2022):')
print(f'  Overall Match Rate: {validation["train_match_rate"]:.1f}% ({validation["train_count"]} signals)')
for sig_type in ['Buy', 'Sell', 'Cash']:
    info = validation['train_per_type'][sig_type]
    print(f'  {sig_type}: {info["rate"]:.1f}% ({info["matched"]}/{info["published"]})')

print(f'\nHeld-out Period (2023-2026):')
print(f'  Overall Match Rate: {validation["heldout_match_rate"]:.1f}% ({validation["heldout_count"]} signals)')
for sig_type in ['Buy', 'Sell', 'Cash']:
    info = validation['heldout_per_type'][sig_type]
    print(f'  {sig_type}: {info["rate"]:.1f}% ({info["matched"]}/{info["published"]})')

print(f'\nDegradation Check:')
print(f'  Degradation: {validation["degradation_pct"]:.1%} (threshold: 10%)')
status = 'PASS' if validation['degradation_pass'] else 'FAIL'
print(f'  Result: {status}')

## 4. Performance Comparison (PERF-01, PERF-02)

Three-way comparison: MDM V2 vs Buy-and-Hold vs MDM Classic across train, held-out, and full periods.

In [ ]:
comparison = build_comparison_table(v2_results, v2_trades, classic_results, classic_trades)

# Format for display
display_df = comparison.copy()
for col in ['total_return', 'annualized_return', 'max_drawdown']:
    display_df[col] = display_df[col].apply(lambda x: f'{x:.2%}' if pd.notna(x) else 'N/A')
display_df['sharpe_ratio'] = display_df['sharpe_ratio'].apply(lambda x: f'{x:.2f}' if pd.notna(x) else 'N/A')
display_df['win_rate'] = display_df['win_rate'].apply(lambda x: f'{x:.1%}' if pd.notna(x) else 'N/A')
display_df['num_trades'] = display_df['num_trades'].astype(int)

display_df

## 5. Dashboard Visualization

In [ ]:
from matplotlib.lines import Line2D

# Filter to analysis period
analysis_start_ts = pd.Timestamp(ANALYSIS_START)
v2_plot = v2_results[v2_results['date'] >= analysis_start_ts].copy().reset_index(drop=True)
classic_plot = classic_results[classic_results['date'] >= analysis_start_ts].copy().reset_index(drop=True)

# Build equity curves
v2_analyzer = V2PerformanceAnalyzer(v2_plot)
v2_equity = v2_analyzer.equity
bh_equity = v2_plot['close'] / v2_plot['close'].iloc[0]
classic_equity = _compute_classic_equity(classic_plot)
v2_drawdown = v2_analyzer.drawdown_series()
model_signals = extract_model_signals(v2_plot)

dates = v2_plot['date']

fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1, figsize=(16, 12), sharex=True,
    gridspec_kw={'height_ratios': [3, 1, 2]}
)

# Top: Equity curves
ax1.plot(dates, v2_equity.values, label='MDM V2', color='blue', linewidth=1.5)
ax1.plot(dates, bh_equity.values, label='Buy-and-Hold', color='gray', linewidth=1.0, alpha=0.7)
ax1.plot(dates, classic_equity.values, label='MDM Classic', color='orange', linewidth=1.0, alpha=0.7)
ax1.set_ylabel('Equity (normalized)')
ax1.set_title('MDM v2 Validation: Equity Curves')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

boundary = pd.Timestamp(HELDOUT_START)
for ax in [ax1, ax2, ax3]:
    ax.axvline(boundary, color='purple', linestyle='--', alpha=0.5, linewidth=1)
ax1.text(boundary, ax1.get_ylim()[1] * 0.95, ' Train | Held-out', color='purple', fontsize=9, va='top')

# Middle: V2 drawdown
ax2.fill_between(dates, v2_drawdown.values, 0, color='red', alpha=0.3)
ax2.plot(dates, v2_drawdown.values, color='red', linewidth=0.8)
ax2.set_ylabel('Drawdown')
ax2.grid(True, alpha=0.3)

# Bottom: Price with signals
ax3.plot(dates, v2_plot['close'].values, color='black', linewidth=0.8, label='NASDAQ')
pub_in_range = published_signals[pd.to_datetime(published_signals['date']) >= analysis_start_ts]
for _, row in pub_in_range.iterrows():
    sig_date = pd.Timestamp(row['date'])
    price_row = v2_plot[v2_plot['date'] == sig_date]
    if price_row.empty:
        date_diffs = (v2_plot['date'] - sig_date).abs()
        nearest_idx = date_diffs.idxmin()
        price = v2_plot.loc[nearest_idx, 'close']
    else:
        price = price_row.iloc[0]['close']
    marker = {"Buy": ('^', 'green', 10), "Sell": ('v', 'red', 10), "Cash": ('o', 'gray', 7)}
    m, c, s = marker.get(row['signal'], ('o', 'black', 5))
    ax3.plot(sig_date, price, m, color=c, markersize=s, alpha=0.8, zorder=5)

ax3.set_ylabel('NASDAQ')
ax3.set_title('Price with Published + Model Signals')
ax3.grid(True, alpha=0.3)

legend_elements = [
    Line2D([0], [0], marker='^', color='w', markerfacecolor='green', markersize=10, label='Published Buy'),
    Line2D([0], [0], marker='v', color='w', markerfacecolor='red', markersize=10, label='Published Sell'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=7, label='Published Cash'),
]
ax3.legend(handles=legend_elements, loc='upper left', fontsize=8, ncol=3)

plt.tight_layout()
plt.show()

## 6. Detailed Exploration

Modify the cells below to explore specific aspects of the validation results.
For example, filter to the held-out period only or examine specific signals.

In [ ]:
# Example: Held-out period equity curve only (2023+)
heldout_start = pd.Timestamp('2023-01-01')
v2_heldout = v2_results[v2_results['date'] >= heldout_start].copy().reset_index(drop=True)

heldout_analyzer = V2PerformanceAnalyzer(v2_heldout, v2_trades)
heldout_summary = heldout_analyzer.summary()

print('Held-out Period (2023+) V2 Performance:')
for k, v in heldout_summary.items():
    if k != 'equity_curve':
        if isinstance(v, float):
            print(f'  {k}: {v:.4f}')
        else:
            print(f'  {k}: {v}')

# Plot held-out equity
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(v2_heldout['date'], heldout_analyzer.equity.values, label='MDM V2', color='blue')
bh_heldout = v2_heldout['close'] / v2_heldout['close'].iloc[0]
ax.plot(v2_heldout['date'], bh_heldout.values, label='Buy-and-Hold', color='gray', alpha=0.7)
ax.set_title('Held-out Period: V2 vs Buy-and-Hold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()